<a href="https://colab.research.google.com/github/ankurs190/Agentic-2.0/blob/main/12_July_Day_19_Autogen_Tools_%26_Teams.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [58]:
!pip install -qU  python-dotenv autogen_agentchat autogen_ext langchain_community
 # write all API Keys first

### **Inbuilt Tool (HttpTool Request)**

In [2]:
!pip install -qU "autogen-agentchat" "autogen-ext[http-tool]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 4.9 MB/s eta 0:00:00


In [3]:
#https://microsoft.github.io/autogen/stable/reference/python/autogen_ext.tools.http.html#autogen_ext.tools.http.HttpTool
!pip install -qU "autogen-agentchat" "autogen-ext[http-tool]"
from autogen_ext.tools.http import HttpTool


# https://catfact.ninja/fact
# {
#   "fact": "Cats spend nearly 1/3 of their waking hours cleaning themselves.",
#   "length": 64
# }
#  JSON Schema (mandatory for httprequesttool) for Cat Fact ( from chatgpt)
schema= {
  "type": "object",
  "properties": {
    "fact": {
      "type": "string",
      "description": "A randomly generated cat fact."
    },
    "length": {
      "type": "integer",
      "description": "The length of the cat fact string."
    }
  },
  "required": ["fact", "length"]
}

# http request from Autogen #https://microsoft.github.io/autogen/stable/reference/python/autogen_ext.tools.http.html#autogen_ext.tools.http.HttpTool
http_tool = HttpTool(
    name="cat_facts_api",
    description="facts about cats",
    scheme="https",
    host="catfact.ninja",
    port=443,
    path="/fact",
    method="GET",
    return_type="json",
    json_schema=schema,
)

In [6]:
# https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/tutorial/models.html
# !pip install "autogen-ext[openai]"

from autogen_ext.models.openai import OpenAIChatCompletionClient
model_client = OpenAIChatCompletionClient(model='gpt-4o',api_key=OPENAI_API_KEY)

In [7]:
import asyncio
from autogen_agentchat.agents import AssistantAgent


 # Create an assistant with the http_tool
model = OpenAIChatCompletionClient(model='gpt-4o',api_key=OPENAI_API_KEY)
agent = AssistantAgent(name= "http_tool_assistant",
                               model_client=model,
                               tools=[http_tool],
                               reflect_on_tool_use=True,
                               system_message='You are a helpful assistant that can provide cat facts using the cat_facts_api tool. Give the result with summary',
                              )

    # The assistant can now use the http_tool tool to get the random facts about cat

# response = await assistant.run(task = 'Give me a random cat fact, please?')

async def main():
    result = await agent.run(task = 'Give me a random cat fact, please?')
    print(result.messages[-1].content)
await main()


Here's a random cat fact: "A cat will tremble or shiver when it is in extreme pain." This behavior can be a sign that the cat is experiencing discomfort or severe pain, and it may require attention or care.


### **3rd Party Tools (Langchain Tools)**

In [9]:
!pip install -qU langchain_community
from langchain_community.utilities import GoogleSerperAPIWrapper
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent

model_client=OpenAIChatCompletionClient(model='gpt-4o',api_key=OPENAI_API_KEY)

search_tool_wrapper = GoogleSerperAPIWrapper(type='search',serper_api_key=SERPER_API_KEY)

def search_web(query:str) ->str:
    """Search the web for the given query and return the results."""

    if (query == 'ipl'):
        return 'IPL is Indian Premier League, a professional Twenty20 cricket league in India.' # Mocking the call
    try:
        results = search_tool_wrapper.run(query)
        return results
    except Exception as e:
        print(f"Error occurred while searching the web: {e}")
        return "No results found."


search_agent = AssistantAgent(
    name="SearchAgent",
    model_client=model_client,
    tools=[search_web],
    description="An agent that can search the web for information.",
    system_message="You are a helpful assistant that can search the web for information using the search_web tool." \
    "Please make sure that you use the search_web tool to find information before you return the answer." \
    "don't send the year in query, rather use latest or recently etc.",
    reflect_on_tool_use=True,
)

async def run_serper_search():
    """Run the search agent with a sample query."""
    query = "Who won the IPL recently ?"
    print(f"Querying: {query}")


    result = await search_agent.run(task=query)
    print(result.messages[-1].content)


await run_serper_search()

Querying: Who won the IPL recently ?
The Royal Challengers Bangalore won the most recent IPL season.


### **Autogen Teams**
https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/tutorial/teams.html

- Round Robin, selector group chat, swarm, graph flow, semantic kernel

##### **RoundRobin**

In [10]:
import asyncio

from autogen_ext.models.openai import OpenAIChatCompletionClient
model_client = OpenAIChatCompletionClient(model='gpt-4o', api_key=OPENAI_API_KEY)

In [12]:
from autogen_agentchat.agents import AssistantAgent

dsa_solver = AssistantAgent(
    name = 'Complex_DSA_Solver',
    model_client=model_client,
    description='A DSA solver',
    system_message="You give code in python to solve complex DSA problems. Give under 100 words."
)

code_reviewer = AssistantAgent(
    name = 'CODE_REVIEWER',
    model_client=model_client,
    description='A Code Reviewer',
    system_message="You review the code given by the complex_dsa_solver and make sure it is optimized.Give under 10 words. If you feel that the code is fine, please say 'TERMINATE'"
)

code_editor = AssistantAgent(
    name = 'CODE_EDITOR',
    model_client=model_client,
    description='A Code editor',
    system_message="You make the code easy to understand and add comments wherever required.Give under 10 words"
)

In [24]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.messages import TextMessage

team = RoundRobinGroupChat(
    participants=[dsa_solver, code_reviewer, code_editor], #in order, sharing context like actual manual teams
    max_turns=3  # try with 1
)

async def run_team():
    text = TextMessage(content='write a simple code in python to add 2 numbers',source='user')
    result = await team.run(task=text)
    print(result)
    for i in result.messages:
      print(f"{i.source } -> {i.content }")

await run_team()

messages=[TextMessage(id='9ddd6986-a14f-4bfd-afec-bb0d223330b8', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 22, 5, 59, 16, 28308, tzinfo=datetime.timezone.utc), content='write a simple code in python to add 2 numbers', type='TextMessage'), TextMessage(id='aa7ae9c9-251c-4824-8a27-e4969cebc498', source='Complex_DSA_Solver', models_usage=RequestUsage(prompt_tokens=1041, completion_tokens=35), metadata={}, created_at=datetime.datetime(2025, 7, 22, 5, 59, 27, 449261, tzinfo=datetime.timezone.utc), content='```python\n# Simple addition of two numbers\na = 4\nb = 5\nsum = a + b\nprint("The sum is:", sum)\n```', type='TextMessage'), TextMessage(id='c6ef629b-3634-495c-b04b-e748f7ef5003', source='CODE_REVIEWER', models_usage=RequestUsage(prompt_tokens=1137, completion_tokens=3), metadata={}, created_at=datetime.datetime(2025, 7, 22, 5, 59, 28, 27315, tzinfo=datetime.timezone.utc), content='TERMINATE', type='TextMessage'), TextMessage(id='82ca0f5c-939e-4f

In [ ]:
messages=[TextMessage(id='758a6434-0888-49a0-b507-0d51ca608bcf', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 22, 5, 42, 24, 455534, tzinfo=datetime.timezone.utc), content='write a simple code in python to add 2 numbers', type='TextMessage'),
        1.  TextMessage(id='ae7ec04d-d675-4a71-b509-1c4855a454ad', source='Complex_DSA_Solver', models_usage=RequestUsage(prompt_tokens=338, completion_tokens=63), metadata={}, created_at=datetime.datetime(2025, 7, 22, 5, 42, 25, 732134, tzinfo=datetime.timezone.utc), content='```python\n# Function to add two numbers\ndef add_numbers(a, b):\n    return a + b\n\n# Example usage\nnum1 = 10\nnum2 = 15\nresult = add_numbers(num1, num2)\n\n# Print the result\nprint("The sum is:", result)\n```', type='TextMessage'),
        2.  TextMessage(id='5195951e-3e9d-4821-b599-98c41fa22451', source='CODE_REVIEWER', models_usage=RequestUsage(prompt_tokens=356, completion_tokens=3), metadata={}, created_at=datetime.datetime(2025, 7, 22, 5, 42, 26, 207072, tzinfo=datetime.timezone.utc), content='TERMINATE', type='TextMessage'),
        3.  TextMessage(id='1f6dbe14-56e9-4c00-bade-34e17cf82be5', source='CODE_EDITOR', models_usage=RequestUsage(prompt_tokens=393, completion_tokens=13), metadata={}, created_at=datetime.datetime(2025, 7, 22, 5, 42, 26, 814074, tzinfo=datetime.timezone.utc), content='Program terminated. If you have questions, feel free to ask!', type='TextMessage')]
          stop_reason='Maximum number of turns 3 reached.'

A messages list from an AutoGen multi-agent conversation, which ended due
to stop_reason='Maximum number of turns 3 reached.'

| Step | Agent                | Message                                                                |
| ---- | -------------------- | ---------------------------------------------------------------------- |
| 1️⃣  | `user`               | `"write a simple code in python to add 2 numbers"`                     |
| 2️⃣  | `Complex_DSA_Solver` | Provides working Python code for adding two numbers                    |
| 3️⃣  | `CODE_REVIEWER`      | Responds with just: `"TERMINATE"`                                      |
| 4️⃣  | `CODE_EDITOR`        | Says: `"Certainly! Let me know if you need further assistance."`       |
| ⛔    | **System**           | Stops the conversation due to: **`Maximum number of turns 3 reached`** |


In [ ]:
#max turn 1
messages=[TextMessage(id='d830216c-6c4b-4ba0-8210-03c86379c526', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 22, 5, 37, 17, 597950, tzinfo=datetime.timezone.utc), content='write a simple code in python to add 2 numbers', type='TextMessage'),

         1. TextMessage(id='8e75e8aa-53dc-4ba7-910d-8fdb2becab87', source='Complex_DSA_Solver', models_usage=RequestUsage(prompt_tokens=127, completion_tokens=84), metadata={}, created_at=datetime.datetime(2025, 7, 22, 5, 37, 19, 299705, tzinfo=datetime.timezone.utc), content='```python\n# Function to add two numbers\ndef add_numbers(a, b):\n    return a + b\n\n# Taking input from the user\nnum1 = float(input("Enter first number: "))\nnum2 = float(input("Enter second number: "))\n\n# Calculating the sum\nresult = add_numbers(num1, num2)\n\n# Displaying the result\nprint("The sum is:", result)\n```', type='TextMessage')]

          stop_reason='Maximum number of turns 1 reached.'


In [ ]:
#max turns 6
messages=[TextMessage(id='468a26ab-16cc-4b24-9e88-1b7a1ed8062f', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 7, 22, 5, 38, 18, 939699, tzinfo=datetime.timezone.utc), content='write a simple code in python to add 2 numbers', type='TextMessage'),
         1. TextMessage(id='7b7f5cef-f223-4d4f-b101-0486a2320560', source='Complex_DSA_Solver', models_usage=RequestUsage(prompt_tokens=231, completion_tokens=33), metadata={}, created_at=datetime.datetime(2025, 7, 22, 5, 38, 19, 878462, tzinfo=datetime.timezone.utc), content='```python\n# Add two numbers\na = 3\nb = 5\nsum = a + b\nprint("The sum is:", sum)\n```', type='TextMessage'),
         2. TextMessage(id='20708092-0a71-46b4-9231-902fb5fb02b7', source='CODE_REVIEWER', models_usage=RequestUsage(prompt_tokens=205, completion_tokens=3), metadata={}, created_at=datetime.datetime(2025, 7, 22, 5, 38, 21, 160858, tzinfo=datetime.timezone.utc), content='TERMINATE', type='TextMessage'),
         3. TextMessage(id='0808d760-3a1c-4f6a-92c7-221f7289e3ea', source='CODE_EDITOR', models_usage=RequestUsage(prompt_tokens=219, completion_tokens=11), metadata={}, created_at=datetime.datetime(2025, 7, 22, 5, 38, 22, 35332, tzinfo=datetime.timezone.utc), content='Program terminated. Let me know if you need help!', type='TextMessage'),
         4. TextMessage(id='290ead30-2bec-4e77-8ad9-1f2dd91e4235', source='Complex_DSA_Solver', models_usage=RequestUsage(prompt_tokens=298, completion_tokens=20), metadata={}, created_at=datetime.datetime(2025, 7, 22, 5, 38, 24, 552546, tzinfo=datetime.timezone.utc), content="If you need further assistance or have more questions, feel free to ask. I'm here to help!", type='TextMessage'),
         5. TextMessage(id='a9072bdb-cbee-4cee-a926-2014d6cbb2f2', source='CODE_REVIEWER', models_usage=RequestUsage(prompt_tokens=260, completion_tokens=3), metadata={}, created_at=datetime.datetime(2025, 7, 22, 5, 38, 25, 640263, tzinfo=datetime.timezone.utc), content='TERMINATE', type='TextMessage'),
         6. TextMessage(id='c38606b0-41cd-490b-a246-7d3f51554476', source='CODE_EDITOR', models_usage=RequestUsage(prompt_tokens=276, completion_tokens=12), metadata={}, created_at=datetime.datetime(2025, 7, 22, 5, 38, 26, 228478, tzinfo=datetime.timezone.utc), content='Program terminated. If more help is needed, just ask!', type='TextMessage')]
           stop_reason='Maximum number of turns 6 reached.'


In [42]:
# with termination conditions
# https://microsoft.github.io/autogen/stable/user-guide/agentchat-user-guide/tutorial/termination.html

from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.conditions import TextMentionTermination

my_termination = TextMentionTermination(text='TERMINATE')

team = RoundRobinGroupChat(
    participants=[dsa_solver, code_reviewer, code_editor],
    termination_condition=my_termination,  # try comment & un comment this. see difference
    max_turns=14
)


async def run_team():
    task = TextMessage(content='write a simple code in python to add 2 numbers',source='user')

    result = await team.run(task=task)

    for i in result.messages:
        print(f"{i.source} : {i.content}")


    # print(result)

await run_team()


# earlier we mentioned termination text in CODE_REVIEWER also, but didn;t specify TextMentionTermination so it kept running till 14th turns.
#now when it finds TextMentionTermination(text='TERMINATE'), it terminates the code

user : write a simple code in python to add 2 numbers
Complex_DSA_Solver : Certainly! Here's a simple Python code to add two numbers:

```python
def add_numbers(a, b):
    return a + b

# Example usage
num1 = 5
num2 = 3
result = add_numbers(num1, num2)
print("The sum is:", result)
```

This code defines a function `add_numbers` that takes two arguments and returns their sum. The example usage demonstrates adding 5 and 3.
CODE_REVIEWER : TERMINATE


In [39]:
# another example with multiple termination conditions
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.conditions import TextMentionTermination,MaxMessageTermination

my_termination = TextMentionTermination(text='TERMINATE') | MaxMessageTermination(max_messages=5)  # user is also counted in turns

team = RoundRobinGroupChat(
    participants=[dsa_solver, code_reviewer, code_editor],
    termination_condition=my_termination,
    max_turns=6
)

async def run_team():
    task = TextMessage(content='write a code to find median of two sorted arrays',source='user')
    result = await team.run(task=task)

    for i in result.messages:
        print(f"{i.source} : {i.content}")


    # print(result)

await run_team()

user : write a code to find median of two sorted arrays
Complex_DSA_Solver : Certainly! You can find the median of two sorted arrays using a binary search approach with a time complexity of O(log(min(n, m))) as follows:

```python
def findMedianSortedArrays(nums1, nums2):
    if len(nums1) > len(nums2):
        nums1, nums2 = nums2, nums1
    x, y = len(nums1), len(nums2)
    low, high = 0, x

    while low <= high:
        partitionX = (low + high) // 2
        partitionY = (x + y + 1) // 2 - partitionX

        maxX = float('-inf') if partitionX == 0 else nums1[partitionX - 1]
        minX = float('inf') if partitionX == x else nums1[partitionX]

        maxY = float('-inf') if partitionY == 0 else nums2[partitionY - 1]
        minY = float('inf') if partitionY == y else nums2[partitionY]

        if maxX <= minY and maxY <= minX:
            if (x + y) % 2 == 0:
                return (max(maxX, maxY) + min(minX, minY)) / 2
            else:
                return max(maxX, maxY)
  

In [38]:
await team.reset()  # Reset the team for a new task.
# await Console(team.run_stream(task="Write a short poem about the fall season."))  # Stream the messages to the console.

from autogen_agentchat.base import TaskResult

team_2 = RoundRobinGroupChat(
    participants=[dsa_solver, code_reviewer, code_editor],
    termination_condition=my_termination,
    max_turns=6
)

async for message in team_2.run_stream(task="Write a simple Hello world code ?"):  # type: ignore

    print(type(message))
    if isinstance(message, TaskResult):  # This code block is checking whether a message returned from an AutoGen agent team execution is the final result or just a normal message from an agent.
    #The isinstance() function is used to check if an object belongs to a specific class or data type.
        print("Stop Reason:", message.stop_reason)
    else:
        print(message.source,message.content)

<class 'autogen_agentchat.messages.TextMessage'>
user Write a simple Hello world code ?
<class 'autogen_agentchat.messages.TextMessage'>
Complex_DSA_Solver Certainly! Here's a simple "Hello, World!" program in Python:

```python
print("Hello, World!")
```
<class 'autogen_agentchat.messages.TextMessage'>
CODE_REVIEWER TERMINATE
<class 'autogen_agentchat.base._task.TaskResult'>
Stop Reason: Text 'TERMINATE' mentioned


In [43]:
state = await team.save_state() # await team.save_state() is used in Microsoft AutoGen's Team class to persist the current conversation state to disk, so you can pause and resume later.
state

{'type': 'TeamState',
 'version': '1.0.0',
 'agent_states': {'Complex_DSA_Solver': {'type': 'ChatAgentContainerState',
   'version': '1.0.0',
   'agent_state': {'type': 'AssistantAgentState',
    'version': '1.0.0',
    'llm_context': {'messages': [{'content': 'Write a simple Hello world code ?',
       'source': 'user',
       'type': 'UserMessage'},
      {'content': 'Certainly! Here\'s a simple "Hello, World!" program in Python:\n\n```python\nprint("Hello, World!")\n```',
       'thought': None,
       'source': 'Complex_DSA_Solver',
       'type': 'AssistantMessage'},
      {'content': 'write a code to find median of two sorted arrays',
       'source': 'user',
       'type': 'UserMessage'},
      {'content': "Certainly! You can find the median of two sorted arrays using a binary search approach with a time complexity of O(log(min(n, m))) as follows:\n\n```python\ndef findMedianSortedArrays(nums1, nums2):\n    if len(nums1) > len(nums2):\n        nums1, nums2 = nums2, nums1\n    x,

In [45]:
await team.load_state(state)

In [51]:
# Resuming a team
from autogen_agentchat.agents import AssistantAgent
add_1_agent_first = AssistantAgent(
    name = 'add_1_agent_first',
    model_client=model_client,
    system_message="Add 1 to the number, first number is 0. Give result as output"
)

add_1_agent_second = AssistantAgent(
    name = 'add_1_agent_second',
    model_client=model_client,
    system_message="Add 1 to the number you got from previous run. Give result as output."
)

add_1_agent_third = AssistantAgent(
    name = 'add_1_agent_third',
    model_client=model_client,
    system_message="Add 1 to the number from previous run. Give result as output."
)

my_increment_team = RoundRobinGroupChat(participants=[add_1_agent_first,add_1_agent_second,add_1_agent_third],max_turns=2)

from autogen_agentchat.ui import Console # Stream conversation output to the terminal or notebook. Visualize agent messages as they happen, nicely formatted
await Console(my_increment_team.run_stream())  #1,2

---------- TextMessage (add_1_agent_first) ----------
1
---------- TextMessage (add_1_agent_second) ----------
2


TaskResult(messages=[TextMessage(id='9af5262d-5637-450c-8c2c-2178c6402def', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=24, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 22, 6, 53, 50, 404009, tzinfo=datetime.timezone.utc), content='1', type='TextMessage'), TextMessage(id='0f3ece08-1a33-4407-b8bb-e583d0a3cdf6', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=34, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 22, 6, 53, 50, 996203, tzinfo=datetime.timezone.utc), content='2', type='TextMessage')], stop_reason='Maximum number of turns 2 reached.')

In [52]:
# autogen team is resuming
await Console(my_increment_team.run_stream()) #3,4

---------- TextMessage (add_1_agent_third) ----------
3
---------- TextMessage (add_1_agent_first) ----------
4


TaskResult(messages=[TextMessage(id='720deee9-ed4b-4183-921c-0ca0c95c66cf', source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=42, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 22, 6, 53, 53, 199265, tzinfo=datetime.timezone.utc), content='3', type='TextMessage'), TextMessage(id='8b530283-2e9d-49b5-a8a6-b940d74d6bf8', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=50, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 22, 6, 53, 53, 615496, tzinfo=datetime.timezone.utc), content='4', type='TextMessage')], stop_reason='Maximum number of turns 2 reached.')

In [53]:
await Console(my_increment_team.run_stream()) # 5,6, see how context is being shared

---------- TextMessage (add_1_agent_second) ----------
5
---------- TextMessage (add_1_agent_third) ----------
6


TaskResult(messages=[TextMessage(id='773a51c6-e9d0-43a6-831b-fe72bc912cc5', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=60, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 22, 6, 53, 54, 995074, tzinfo=datetime.timezone.utc), content='5', type='TextMessage'), TextMessage(id='f60a9ccf-93a8-4d0c-b788-c12c5415e531', source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=67, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 22, 6, 53, 55, 374204, tzinfo=datetime.timezone.utc), content='6', type='TextMessage')], stop_reason='Maximum number of turns 2 reached.')

In [55]:
# to reset team
await my_increment_team.reset()
await Console(my_increment_team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
1
---------- TextMessage (add_1_agent_second) ----------
2


TaskResult(messages=[TextMessage(id='c862d179-4c1e-411f-92be-cd425e832a60', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=24, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 22, 6, 55, 1, 831447, tzinfo=datetime.timezone.utc), content='1', type='TextMessage'), TextMessage(id='63b7d358-d37a-4d8b-b6f6-354ae86bea93', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=34, completion_tokens=1), metadata={}, created_at=datetime.datetime(2025, 7, 22, 6, 55, 2, 339572, tzinfo=datetime.timezone.utc), content='2', type='TextMessage')], stop_reason='Maximum number of turns 2 reached.')

In [57]:
from autogen_core import CancellationToken  # Imports the cancellation mechanism used across AutoGen agents and teams.
cancellation_token = CancellationToken()  # Creates a new token you can pass into any run(...) or run_stream(...) method.

# Use another coroutine to run the team. Starts the team run as a background task using asyncio.create_task(...).

run = asyncio.create_task(
    team.run(
        task="Translate the poem to Spanish.",
        cancellation_token=cancellation_token,
    )
)

# Cancel the run. Immediately cancels the task (in a real scenario, this could happen via button press, timeout, etc.).
cancellation_token.cancel()

try:
    result = await run  # This will raise a CancelledError.
except asyncio.CancelledError as e:
    print(e)
    print("Task was cancelled.")


Task was cancelled.
